# 🔄 Notebook 4: Read Replicas

When a single database can't handle the read load, distribute reads across multiple servers using read replicas.

## Learning Objectives

By the end of this notebook, you'll understand:
- Leader-follower replication
- Synchronous vs asynchronous replication
- Handling replication lag
- Read-after-write consistency

## 🛠️ Setup

**1. Start PostgreSQL + Redis + visualization tools** (from the lab root):

```bash
cd 04-patterns/scaling-reads
docker compose up -d
uv sync
```

**2. Select the `.venv` kernel**

Click the kernel picker in the top-right of this notebook and choose the
`.venv` Python interpreter. If it doesn't appear, reload the VS Code window
(`Cmd+Shift+P` → "Developer: Reload Window") and try again.

### 🔍 Visualization tools (optional but recommended)

| Tool | URL | Use it to |
|------|-----|-----------|
| Adminer (PostgreSQL) | http://localhost:8080 | See tables, run SQL, view execution plans |
| RedisInsight | http://localhost:5540 | Watch cache keys, TTLs, hits/misses |

**Adminer login**: System `PostgreSQL`, Server `postgres`, User `demo`,
Password `demo`, Database `scaling_demo`.


In [1]:
import time
import random
from dataclasses import dataclass
from typing import Optional
from concurrent.futures import ThreadPoolExecutor

print("✅ Ready to learn about read replicas!")

✅ Ready to learn about read replicas!


## 🏗️ Leader-Follower Architecture

In [2]:
print("🏗️ Leader-Follower Replication")
print("=" * 60)
print("""
                    ┌─────────────────┐
                    │  Application    │
                    └────────┬────────┘
                             │
              ┌──────────────┴──────────────┐
              │                             │
        WRITES│                       READS │
              ▼                             ▼
    ┌─────────────────┐         ┌─────────────────┐
    │     LEADER      │         │   FOLLOWERS     │
    │    (Primary)    │────────►│   (Replicas)    │
    │                 │ replicate│                 │
    │  • All writes   │         │  • Read only    │
    │  • Source of    │         │  • Multiple     │
    │    truth        │         │    servers      │
    └─────────────────┘         └─────────────────┘

KEY POINTS:
─────────────────────────────────────────────────────────────
• Writes ONLY go to leader
• Leader replicates changes to followers
• Reads distributed across ALL followers
• Add more followers = handle more reads
""")

🏗️ Leader-Follower Replication

                    ┌─────────────────┐
                    │  Application    │
                    └────────┬────────┘
                             │
              ┌──────────────┴──────────────┐
              │                             │
        WRITES│                       READS │
              ▼                             ▼
    ┌─────────────────┐         ┌─────────────────┐
    │     LEADER      │         │   FOLLOWERS     │
    │    (Primary)    │────────►│   (Replicas)    │
    │                 │ replicate│                 │
    │  • All writes   │         │  • Read only    │
    │  • Source of    │         │  • Multiple     │
    │    truth        │         │    servers      │
    └─────────────────┘         └─────────────────┘

KEY POINTS:
─────────────────────────────────────────────────────────────
• Writes ONLY go to leader
• Leader replicates changes to followers
• Reads distributed across ALL followers
• Add more followers = handle mo

## 🔄 Simulating Replication

In [3]:
@dataclass
class Database:
    name: str
    is_leader: bool
    data: dict
    replication_lag_ms: float = 0
    
    def read(self, key: str) -> Optional[str]:
        time.sleep(0.001)
        return self.data.get(key)
    
    def write(self, key: str, value: str) -> bool:
        if not self.is_leader:
            raise Exception("Cannot write to replica!")
        time.sleep(0.002)
        self.data[key] = value
        return True

class ReplicatedDatabase:
    def __init__(self, num_replicas: int = 3, replication_lag_ms: float = 50):
        self.leader = Database("leader", True, {})
        self.replicas = [
            Database(f"replica-{i}", False, {}, replication_lag_ms)
            for i in range(num_replicas)
        ]
        self.replication_lag_ms = replication_lag_ms
        self.pending_replications = []
    
    def write(self, key: str, value: str):
        self.leader.write(key, value)
        write_time = time.time()
        self.pending_replications.append((key, value, write_time))
        return True
    
    def _apply_replications(self):
        current_time = time.time()
        still_pending = []
        
        for key, value, write_time in self.pending_replications:
            elapsed_ms = (current_time - write_time) * 1000
            if elapsed_ms >= self.replication_lag_ms:
                for replica in self.replicas:
                    replica.data[key] = value
            else:
                still_pending.append((key, value, write_time))
        
        self.pending_replications = still_pending
    
    def read_from_replica(self, key: str) -> tuple:
        self._apply_replications()
        replica = random.choice(self.replicas)
        value = replica.read(key)
        return value, replica.name
    
    def read_from_leader(self, key: str) -> tuple:
        value = self.leader.read(key)
        return value, "leader"

db = ReplicatedDatabase(num_replicas=3, replication_lag_ms=100)
print("✅ Replicated database created with 3 replicas")
print(f"   Simulated replication lag: 100ms")

✅ Replicated database created with 3 replicas
   Simulated replication lag: 100ms


In [4]:
print("🔄 Demonstrating Replication Lag")
print("=" * 60)

db.write("user:1:name", "Alice")
print("\n✏️ Wrote 'Alice' to leader")

print("\n📖 Reading immediately after write:")
for i in range(3):
    value, source = db.read_from_replica("user:1:name")
    status = "✅" if value else "❌ STALE"
    print(f"   Read from {source}: {value or 'None'} {status}")

print("\n⏳ Waiting for replication (150ms)...")
time.sleep(0.15)

print("\n📖 Reading after replication lag:")
for i in range(3):
    value, source = db.read_from_replica("user:1:name")
    status = "✅" if value else "❌ STALE"
    print(f"   Read from {source}: {value or 'None'} {status}")

🔄 Demonstrating Replication Lag

✏️ Wrote 'Alice' to leader

📖 Reading immediately after write:
   Read from replica-2: None ❌ STALE
   Read from replica-1: None ❌ STALE
   Read from replica-1: None ❌ STALE

⏳ Waiting for replication (150ms)...



📖 Reading after replication lag:
   Read from replica-0: Alice ✅
   Read from replica-1: Alice ✅
   Read from replica-2: Alice ✅


## ⚠️ The Read-After-Write Problem

In [5]:
print("⚠️ Read-After-Write Consistency Problem")
print("=" * 60)
print("""
SCENARIO: User updates their profile
─────────────────────────────────────────────────────────────

1. User changes name from "Alice" to "Alicia"
   └─► Write goes to LEADER

2. Page refreshes to show updated profile
   └─► Read goes to REPLICA (still has "Alice"!)

3. User sees OLD name "Alice" 😱
   └─► Thinks the update failed!

─────────────────────────────────────────────────────────────
""")

db2 = ReplicatedDatabase(num_replicas=3, replication_lag_ms=100)
db2.write("user:1:name", "Alice")
time.sleep(0.15)

print("\nSimulating the problem:")
print("1. User updates name to 'Alicia'...")
db2.write("user:1:name", "Alicia")

print("2. Page refreshes, reading from replica...")
value, source = db2.read_from_replica("user:1:name")
print(f"   Got: '{value}' from {source}")
print(f"   ❌ User sees old name!")

⚠️ Read-After-Write Consistency Problem

SCENARIO: User updates their profile
─────────────────────────────────────────────────────────────

1. User changes name from "Alice" to "Alicia"
   └─► Write goes to LEADER

2. Page refreshes to show updated profile
   └─► Read goes to REPLICA (still has "Alice"!)

3. User sees OLD name "Alice" 😱
   └─► Thinks the update failed!

─────────────────────────────────────────────────────────────




Simulating the problem:
1. User updates name to 'Alicia'...
2. Page refreshes, reading from replica...
   Got: 'Alice' from replica-0
   ❌ User sees old name!


## 🛡️ Solutions for Read-After-Write

In [6]:
print("🛡️ Solutions for Read-After-Write Consistency")
print("=" * 60)
print("""
SOLUTION 1: Read Your Own Writes from Leader
─────────────────────────────────────────────────────────────
• After write, read from leader for that user's session
• Track "last write timestamp" per user
• If recent write, read from leader; else read from replica

SOLUTION 2: Synchronous Replication
─────────────────────────────────────────────────────────────
• Wait for at least one replica to confirm
• Slower writes, but guaranteed consistency
• Trade-off: higher latency for writes

SOLUTION 3: Client-Side Optimistic Updates
─────────────────────────────────────────────────────────────
• Update UI immediately after write
• Don't re-fetch from database
• Simple but doesn't solve server-side reads

SOLUTION 4: Causal Consistency Token
─────────────────────────────────────────────────────────────
• Return write timestamp with response
• Client sends timestamp with next read
• Server waits for replica to catch up to that timestamp
""")

🛡️ Solutions for Read-After-Write Consistency

SOLUTION 1: Read Your Own Writes from Leader
─────────────────────────────────────────────────────────────
• After write, read from leader for that user's session
• Track "last write timestamp" per user
• If recent write, read from leader; else read from replica

SOLUTION 2: Synchronous Replication
─────────────────────────────────────────────────────────────
• Wait for at least one replica to confirm
• Slower writes, but guaranteed consistency
• Trade-off: higher latency for writes

SOLUTION 3: Client-Side Optimistic Updates
─────────────────────────────────────────────────────────────
• Update UI immediately after write
• Don't re-fetch from database
• Simple but doesn't solve server-side reads

SOLUTION 4: Causal Consistency Token
─────────────────────────────────────────────────────────────
• Return write timestamp with response
• Client sends timestamp with next read
• Server waits for replica to catch up to that timestamp



In [7]:
class SmartReplicatedDatabase(ReplicatedDatabase):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.user_last_writes = {}
    
    def write_for_user(self, user_id: str, key: str, value: str):
        self.write(key, value)
        self.user_last_writes[user_id] = time.time()
        return True
    
    def read_for_user(self, user_id: str, key: str) -> tuple:
        last_write = self.user_last_writes.get(user_id, 0)
        time_since_write = (time.time() - last_write) * 1000
        
        if time_since_write < self.replication_lag_ms * 2:
            return self.read_from_leader(key)
        else:
            return self.read_from_replica(key)

print("🛡️ Smart Database with Read-Your-Own-Writes")
print("=" * 60)

smart_db = SmartReplicatedDatabase(num_replicas=3, replication_lag_ms=100)
smart_db.write("user:1:name", "Alice")
time.sleep(0.15)

print("\n1. User updates name to 'Alicia'...")
smart_db.write_for_user("user-1", "user:1:name", "Alicia")

print("2. Page refreshes with smart routing...")
value, source = smart_db.read_for_user("user-1", "user:1:name")
print(f"   Got: '{value}' from {source}")
print(f"   ✅ Routed to leader for recent writer!")

print("\n3. Another user reads same data...")
value, source = smart_db.read_for_user("user-2", "user:1:name")
print(f"   Got: '{value}' from {source}")
print(f"   ✅ Routed to replica (didn't write recently)")

🛡️ Smart Database with Read-Your-Own-Writes



1. User updates name to 'Alicia'...


2. Page refreshes with smart routing...
   Got: 'Alicia' from leader
   ✅ Routed to leader for recent writer!

3. Another user reads same data...
   Got: 'Alice' from replica-1
   ✅ Routed to replica (didn't write recently)


## 📊 Sync vs Async Replication

In [8]:
print("📊 Synchronous vs Asynchronous Replication")
print("=" * 60)
print("""
ASYNCHRONOUS (Default)
─────────────────────────────────────────────────────────────
    Client ──► Leader ──► Response
                  │
                  └──► Replicas (eventually)

• Write returns immediately
• Replicas catch up in background
• Risk: Data loss if leader fails before replication
• Pro: Faster writes

─────────────────────────────────────────────────────────────

SYNCHRONOUS
─────────────────────────────────────────────────────────────
    Client ──► Leader ──► Replicas ──► ACK ──► Response

• Write waits for replica confirmation
• Guaranteed to be on at least 2 servers
• Risk: Slower writes, replica failure blocks writes
• Pro: No data loss

─────────────────────────────────────────────────────────────

SEMI-SYNCHRONOUS (Common in production)
─────────────────────────────────────────────────────────────
• Wait for ONE replica to confirm
• Other replicas are async
• Balance between safety and speed
""")

📊 Synchronous vs Asynchronous Replication

ASYNCHRONOUS (Default)
─────────────────────────────────────────────────────────────
    Client ──► Leader ──► Response
                  │
                  └──► Replicas (eventually)

• Write returns immediately
• Replicas catch up in background
• Risk: Data loss if leader fails before replication
• Pro: Faster writes

─────────────────────────────────────────────────────────────

SYNCHRONOUS
─────────────────────────────────────────────────────────────
    Client ──► Leader ──► Replicas ──► ACK ──► Response

• Write waits for replica confirmation
• Guaranteed to be on at least 2 servers
• Risk: Slower writes, replica failure blocks writes
• Pro: No data loss

─────────────────────────────────────────────────────────────

SEMI-SYNCHRONOUS (Common in production)
─────────────────────────────────────────────────────────────
• Wait for ONE replica to confirm
• Other replicas are async
• Balance between safety and speed



## 📈 Scaling with Replicas

In [9]:
def simulate_load(db: ReplicatedDatabase, queries_per_second: int, duration: float):
    queries = int(queries_per_second * duration)
    times = []
    
    for _ in range(queries):
        start = time.time()
        db.read_from_replica("test_key")
        times.append((time.time() - start) * 1000)
    
    return {
        "total": queries,
        "avg_ms": sum(times) / len(times),
        "throughput": queries / duration
    }

print("📈 Scaling Reads with Replicas")
print("=" * 60)
print()

for num_replicas in [1, 3, 5, 10]:
    db = ReplicatedDatabase(num_replicas=num_replicas, replication_lag_ms=50)
    db.write("test_key", "test_value")
    time.sleep(0.1)
    
    results = simulate_load(db, queries_per_second=1000, duration=0.1)
    capacity = num_replicas * 10000
    
    print(f"Replicas: {num_replicas}")
    print(f"   Simulated capacity: ~{capacity:,} reads/sec")
    print(f"   Avg latency: {results['avg_ms']:.2f}ms")
    print()

📈 Scaling Reads with Replicas



Replicas: 1
   Simulated capacity: ~10,000 reads/sec
   Avg latency: 3.77ms



Replicas: 3
   Simulated capacity: ~30,000 reads/sec
   Avg latency: 4.09ms



Replicas: 5
   Simulated capacity: ~50,000 reads/sec
   Avg latency: 4.22ms



Replicas: 10
   Simulated capacity: ~100,000 reads/sec
   Avg latency: 4.12ms



## 🧪 Quick Quiz

1. **Why can't you write to a replica?**

2. **What's the read-after-write problem?**

3. **When would you use synchronous replication?**

In [10]:
print("📝 Quiz Answers")
print("=" * 50)
print()
print("1. Why no writes to replicas:")
print("   - Creates conflict (which write wins?)")
print("   - Leader is source of truth")
print("   - Replication is one-way: leader → follower")
print()
print("2. Read-after-write problem:")
print("   - User writes to leader")
print("   - Immediately reads from replica")
print("   - Replica hasn't received write yet")
print("   - User sees stale (old) data")
print()
print("3. When to use sync replication:")
print("   - Critical data (financial, medical)")
print("   - When data loss is unacceptable")
print("   - Willing to accept slower writes")

📝 Quiz Answers

1. Why no writes to replicas:
   - Creates conflict (which write wins?)
   - Leader is source of truth
   - Replication is one-way: leader → follower

2. Read-after-write problem:
   - User writes to leader
   - Immediately reads from replica
   - Replica hasn't received write yet
   - User sees stale (old) data

3. When to use sync replication:
   - Critical data (financial, medical)
   - When data loss is unacceptable
   - Willing to accept slower writes


## 🐘 Doing This for Real in PostgreSQL

So far we simulated replication in Python. That was great for understanding
the *concepts*, but in production you use your database's built-in replication.
PostgreSQL offers two flavours:

### 1. Streaming replication (physical)

The replica is a **byte-for-byte copy** of the primary. Postgres ships its
Write-Ahead Log (WAL) to the replica, which replays it.

**Minimal setup** (primary's `postgresql.conf`):

```conf
wal_level = replica            # needed so the WAL contains enough info
max_wal_senders = 10           # how many replicas can stream at once
```

**On the primary** — create a replication user:

```sql
CREATE ROLE repuser WITH REPLICATION LOGIN PASSWORD 'secret';
```

**On the replica** — clone the primary, then start it as a standby:

```bash
# One-time: copy the entire data directory from primary.
pg_basebackup -h primary-host -U repuser -D /var/lib/postgresql/data -Fp -Xs -P -R
# The -R flag writes a standby.signal file so Postgres boots in replica mode.

# Then just start Postgres. It will connect to the primary and stream WAL.
```

**Pros**: automatic, whole-cluster, lowest lag.
**Cons**: replica is read-only, must be same major version, can't replicate a subset.

### 2. Logical replication (selective)

Replicate *specific tables* between databases — even across major versions.
Uses `PUBLICATION` (on primary) + `SUBSCRIPTION` (on replica).

```sql
-- On primary:
CREATE PUBLICATION posts_pub FOR TABLE posts, users;

-- On replica (different cluster, possibly different version):
CREATE SUBSCRIPTION posts_sub
  CONNECTION 'host=primary-host dbname=scaling_demo user=repuser password=secret'
  PUBLICATION posts_pub;
```

**Great for**: analytics replicas, zero-downtime upgrades, replicating only the
hot tables to a cheaper read-only instance.

### 📏 Monitoring replication lag (the #1 thing you'll debug)

Once replication is running, the single most important query is:

```sql
-- Run this ON THE PRIMARY:
SELECT
    client_addr,
    state,
    sent_lsn,                 -- how much WAL we've sent
    replay_lsn,               -- how much the replica has applied
    pg_wal_lsn_diff(sent_lsn, replay_lsn) AS lag_bytes,
    write_lag, flush_lag, replay_lag  -- time-based lag
FROM pg_stat_replication;
```

If `replay_lag` grows to seconds or minutes, your replica is falling behind —
that's when read-after-write problems start biting users.

### 🧰 Who actually sets this up?

In real life, almost nobody configures streaming replication by hand. You use:

| Tool | What it gives you |
|------|-------------------|
| **Amazon RDS / Aurora** | One click: "Create read replica" |
| **Google Cloud SQL** | Same, via `--master-instance-name` flag |
| **Patroni** + etcd | Self-managed HA with automatic failover |
| **Bitnami Postgres-HA Helm chart** | Kubernetes-native primary + replicas |

The *concepts* in this notebook (leader/follower, lag, read-your-own-writes)
are what you apply to each of these — they all have the same failure modes.


## 📚 Summary

### Key Takeaways

1. **Leader handles writes, replicas handle reads** - simple split
2. **Replication lag is unavoidable** - plan for it
3. **Read-your-own-writes** - route recent writers to leader
4. **Add replicas to scale reads** - linear scaling
5. **Sync vs async** - trade-off between safety and speed

### Next Up

In **Notebook 5**, we'll learn about application caching:
- Redis as a cache layer
- TTL strategies
- Cache invalidation